In [1]:
import pandas as pd 

cust_dim = pd.read_csv("customer_dimension.csv")
cust_dim.head()

,customer_id,name,city,plan,effective_start,effective_end,is_current
0,101,Rahul,Pune,GOLD,2026-01-01,2026-06-30,N
1,101,Rahul,Mumbai,GOLD,2026-07-01,NaN,Y
2,102,Neha,Delhi,SILVER,2026-02-15,NaN,Y
3,103,Amit,Bangalore,GOLD,2026-03-10,NaN,Y
4,105,Karan,Chennai,BRONZE,2026-04-01,NaN,Y


In [2]:
cust_snap = pd.read_csv("customer_snapshot.csv")
cust_snap.head()

,customer_id,name,city,plan
0,101,Rahul,Bangalore,GOLD
1,102,Neha,Delhi,SILVER
2,103,Amit,Bangalore,PLATINUM
3,104,Priya,Pune,SILVER
4,105,Karan,Chennai,BRONZE


In [3]:
cust_dim["effective_start"] = pd.to_datetime(cust_dim["effective_start"],errors = "coerce")

In [4]:
cust_dim.dtypes

customer_id                 int64
name                       object
city                       object
plan                       object
effective_start    datetime64[ns]
effective_end              object
is_current                 object
dtype: object

In [5]:
cust_dim["effective_end"] = pd.to_datetime(cust_dim["effective_end"],errors = "coerce")
cust_dim.head()

,customer_id,name,city,plan,effective_start,effective_end,is_current
0,101,Rahul,Pune,GOLD,2026-01-01,2026-06-30,N
1,101,Rahul,Mumbai,GOLD,2026-07-01,NaT,Y
2,102,Neha,Delhi,SILVER,2026-02-15,NaT,Y
3,103,Amit,Bangalore,GOLD,2026-03-10,NaT,Y
4,105,Karan,Chennai,BRONZE,2026-04-01,NaT,Y


In [6]:
current_records = cust_dim[cust_dim["is_current"] == "Y"]
current_records.head()

,customer_id,name,city,plan,effective_start,effective_end,is_current
1,101,Rahul,Mumbai,GOLD,2026-07-01,NaT,Y
2,102,Neha,Delhi,SILVER,2026-02-15,NaT,Y
3,103,Amit,Bangalore,GOLD,2026-03-10,NaT,Y
4,105,Karan,Chennai,BRONZE,2026-04-01,NaT,Y


In [8]:
snapshot_merged  = cust_snap.merge(current_records,on = "customer_id",how = "left",suffixes= ("_snap","_current"))
snapshot_merged.head()

,customer_id,name_snap,city_snap,plan_snap,name_current,city_current,plan_current,effective_start,effective_end,is_current
0,101,Rahul,Bangalore,GOLD,Rahul,Mumbai,GOLD,2026-07-01,NaT,Y
1,102,Neha,Delhi,SILVER,Neha,Delhi,SILVER,2026-02-15,NaT,Y
2,103,Amit,Bangalore,PLATINUM,Amit,Bangalore,GOLD,2026-03-10,NaT,Y
3,104,Priya,Pune,SILVER,NaN,NaN,NaN,NaT,NaT,NaN
4,105,Karan,Chennai,BRONZE,Karan,Chennai,BRONZE,2026-04-01,NaT,Y


In [13]:
new_records = snapshot_merged[snapshot_merged["is_current"].isna()]
new_records

,customer_id,name_snap,city_snap,plan_snap,name_current,city_current,plan_current,effective_start,effective_end,is_current
3,104,Priya,Pune,SILVER,NaN,NaN,NaN,NaT,NaT,NaN


In [16]:
changed_records = snapshot_merged[
    (~snapshot_merged["is_current"].isna()) & 
    (
        (snapshot_merged["city_snap"] != snapshot_merged["city_current"]) |
    (snapshot_merged["plan_snap"] != snapshot_merged["plan_current"])
    )
    ]
changed_records.head()

,customer_id,name_snap,city_snap,plan_snap,name_current,city_current,plan_current,effective_start,effective_end,is_current
0,101,Rahul,Bangalore,GOLD,Rahul,Mumbai,GOLD,2026-07-01,NaT,Y
2,103,Amit,Bangalore,PLATINUM,Amit,Bangalore,GOLD,2026-03-10,NaT,Y


In [12]:
unchanged_records = snapshot_merged[
    (snapshot_merged["city_snap"] == snapshot_merged["city_current"]) &
    (snapshot_merged["plan_snap"] == snapshot_merged["plan_current"])
    ]

unchanged_records.head()

,customer_id,name_snap,city_snap,plan_snap,name_current,city_current,plan_current,effective_start,effective_end,is_current
1,102,Neha,Delhi,SILVER,Neha,Delhi,SILVER,2026-02-15,NaT,Y
4,105,Karan,Chennai,BRONZE,Karan,Chennai,BRONZE,2026-04-01,NaT,Y


In [18]:
cust_dim.loc[(cust_dim["customer_id"].isin(changed_records["customer_id"])) & (cust_dim["is_current"] == "Y"),"effective_end"] = "2026-09-30"

In [19]:
cust_dim.loc[(cust_dim["customer_id"].isin(changed_records["customer_id"])) & (cust_dim["is_current"] == "Y"),"is_current"] = "N"

In [20]:
cust_dim

,customer_id,name,city,plan,effective_start,effective_end,is_current
0,101,Rahul,Pune,GOLD,2026-01-01,2026-06-30,N
1,101,Rahul,Mumbai,GOLD,2026-07-01,2026-09-30,N
2,102,Neha,Delhi,SILVER,2026-02-15,NaT,Y
3,103,Amit,Bangalore,GOLD,2026-03-10,2026-09-30,N
4,105,Karan,Chennai,BRONZE,2026-04-01,NaT,Y


In [21]:
cust_dim.loc[(cust_dim["customer_id"].isin(changed_records["customer_id"])) & (cust_dim["is_current"] == "Y"),"effective_end"] = "2026-09-30"

In [25]:
records_to_insert = pd.concat([changed_records,new_records],ignore_index=True)

In [26]:
records_to_insert

,customer_id,name_snap,city_snap,plan_snap,name_current,city_current,plan_current,effective_start,effective_end,is_current
0,101,Rahul,Bangalore,GOLD,Rahul,Mumbai,GOLD,2026-07-01,NaT,Y
1,103,Amit,Bangalore,PLATINUM,Amit,Bangalore,GOLD,2026-03-10,NaT,Y
2,104,Priya,Pune,SILVER,NaN,NaN,NaN,NaT,NaT,NaN


In [28]:
records_to_insert = records_to_insert.drop(["name_current","city_current","plan_current"],axis= 1
                                           )

In [29]:
records_to_insert

,customer_id,name_snap,city_snap,plan_snap,effective_start,effective_end,is_current
0,101,Rahul,Bangalore,GOLD,2026-07-01,NaT,Y
1,103,Amit,Bangalore,PLATINUM,2026-03-10,NaT,Y
2,104,Priya,Pune,SILVER,NaT,NaT,NaN


In [30]:
records_to_insert.columns

Index(['customer_id', 'name_snap', 'city_snap', 'plan_snap', 'effective_start',
       'effective_end', 'is_current'],
      dtype='object')

In [33]:
records_to_insert = records_to_insert.rename(
    columns= {"name_snap":"name",
              "city_snap":"city",
              "plan_snap":"plan"
              }
)


In [34]:
records_to_insert

,customer_id,name,city,plan,effective_start,effective_end,is_current
0,101,Rahul,Bangalore,GOLD,2026-07-01,NaT,Y
1,103,Amit,Bangalore,PLATINUM,2026-03-10,NaT,Y
2,104,Priya,Pune,SILVER,NaT,NaT,NaN


In [38]:
records_to_insert["effective_start"] = pd.to_datetime("2026-10-01")

In [39]:
records_to_insert["effective_end"] = pd.NaT

In [40]:
records_to_insert["is_current"] = "Y"

In [41]:
records_to_insert[
    ["customer_id", "name", "city", "plan",
     "effective_start", "effective_end", "is_current"]
]

,customer_id,name,city,plan,effective_start,effective_end,is_current
0,101,Rahul,Bangalore,GOLD,2026-10-01,NaT,Y
1,103,Amit,Bangalore,PLATINUM,2026-10-01,NaT,Y
2,104,Priya,Pune,SILVER,2026-10-01,NaT,Y


In [42]:
final_customer_dimension = pd.concat([cust_dim,records_to_insert],ignore_index=True)

In [43]:
final_customer_dimension

,customer_id,name,city,plan,effective_start,effective_end,is_current
0,101,Rahul,Pune,GOLD,2026-01-01,2026-06-30,N
1,101,Rahul,Mumbai,GOLD,2026-07-01,2026-09-30,N
2,102,Neha,Delhi,SILVER,2026-02-15,NaT,Y
3,103,Amit,Bangalore,GOLD,2026-03-10,2026-09-30,N
4,105,Karan,Chennai,BRONZE,2026-04-01,NaT,Y
5,101,Rahul,Bangalore,GOLD,2026-10-01,NaT,Y
6,103,Amit,Bangalore,PLATINUM,2026-10-01,NaT,Y
7,104,Priya,Pune,SILVER,2026-10-01,NaT,Y
